In [8]:
import csv
import json
import logging

logging.basicConfig(level=logging.INFO)

def make_report(csv_path: str, json_path: str) -> int:
    try:
        with open(csv_path, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            
            report_data = []
            count = 0
            
            for row in reader:
                name = row["이름"]
                student_id = row["학번"]
                mid = int(row["중간"]) if row["중간"] else None
                fin = int(row["기말"]) if row["기말"] else None
                hw = int(row["과제"]) if row["과제"] else None
                
            
                if mid is None or fin is None or hw is None:
                    avg = None
                    grade = None
                else:
                    avg = (mid * 0.3) + (fin * 0.5) + (hw * 0.2)
                    grade = "A" if avg >= 90 else "B" if avg >= 80 else "C" if avg >= 70 else "F"
                        
                report_data.append({
                    "이름": name, "학번": student_id,
                    "점수": {"중간": mid, "기말": fin, "과제": hw},
                    "평균": avg, "등급": grade
                })
                
                logging.info(f"이름: {name}, 평균: {avg}, 등급: {grade}")
                count += 1
                
    except FileNotFoundError:
        logging.warning("CSV 파일이 존재하지 않습니다.")
        return 0
    except UnicodeDecodeError:
        logging.error("CSV 파일의 인코딩이 잘못되었습니다.")
        return 0

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(report_data, f, ensure_ascii=False, indent=2)
        
    return count


make_report("scores.csv", "report.json")

INFO:root:이름: 김언어, 평균: 89.5, 등급: B
INFO:root:이름: 이국문, 평균: 84.4, 등급: B
INFO:root:이름: 박영문, 평균: 93.5, 등급: A
INFO:root:이름: 최역사, 평균: None, 등급: None


4

(1) 생성된report.json의전체내용

[
  {
    "이름": "김언어",
    "학번": "2026-10000",
    "점수": {
      "중간": 85,
      "기말": 92,
      "과제": 90
    },
    "평균": 89.5,
    "등급": "B"
  },
  {
    "이름": "이국문",
    "학번": "2026-12345",
    "점수": {
      "중간": 78,
      "기말": 88,
      "과제": 85
    },
    "평균": 84.4,
    "등급": "B"
  },
  {
    "이름": "박영문",
    "학번": "2026-13579",
    "점수": {
      "중간": 95,
      "기말": 90,
      "과제": 100
    },
    "평균": 93.5,
    "등급": "A"
  },
  {
    "이름": "최역사",
    "학번": "2025-11111",
    "점수": {
      "중간": null,
      "기말": 82,
      "과제": 88
    },
    "평균": null,
    "등급": null
  }
]

(2) 화면에 출력되는 logging 메시지

INFO:root:이름: 김언어, 평균: 89.5, 등급: B
INFO:root:이름: 이국문, 평균: 84.4, 등급: B
INFO:root:이름: 박영문, 평균: 93.5, 등급: A
INFO:root:이름: 최역사, 평균: None, 등급: None
4

(3) 설명

한글이 깨지지 않게 모두 encoding="utf-8"을 지정했습니다. 
없는 파일을 마주쳤을 때 프로그램이 멈추지 않도록 try-except로 로깅 처리했습니다.
CSV 파일에서 점수가 비어있으면 DictReader가 빈 문자열로 읽어오는데, 값이 없을 때 none을 할당하고 나머지 점수들은 정수들로 변환하고 평균과 등급에는 none을 할당하여 JSON의 null로 나타나게 하였습니다.

In [9]:
class InvalidJamoError(ValueError):
    pass

def classify_jamo(c: str) -> str:
    if not isinstance(c, str):
        raise TypeError(f"인자가 str 타입이 아닙니다: {c}")
    if len(c) != 1:
        raise ValueError(f"길이가 1이 아닙니다: {c}")
        
    code = ord(c)
    if 0x3131 <= code <= 0x314E: return "자음"
    if 0x314F <= code <= 0x3163: return "모음"
        
    raise InvalidJamoError(f"한글 자음이나 모음이 아닙니다: {c}")

inputs = ["ㄱ", "ㅏ", "가", " ", "AB", 5, "", "ㅠ", "ㅎ"]

for c in inputs:
    try:
        print(f"'{c}' -> {classify_jamo(c)}")
    except TypeError as e:
        print(f"[TypeError] {e}")
    except InvalidJamoError as e:
        print(f"[InvalidJamoError] {e}")
    except ValueError as e:
        print(f"[ValueError] {e}")

'ㄱ' -> 자음
'ㅏ' -> 모음
[InvalidJamoError] 한글 자음이나 모음이 아닙니다: 가
[InvalidJamoError] 한글 자음이나 모음이 아닙니다:  
[ValueError] 길이가 1이 아닙니다: AB
[TypeError] 인자가 str 타입이 아닙니다: 5
[ValueError] 길이가 1이 아닙니다: 
'ㅠ' -> 모음
'ㅎ' -> 자음


(1) input 실행 결과 

'ㄱ' -> 자음
'ㅏ' -> 모음
[InvalidJamoError] 한글 자음이나 모음이 아닙니다: 가
[InvalidJamoError] 한글 자음이나 모음이 아닙니다:  
[ValueError] 길이가 1이 아닙니다: AB
[TypeError] 인자가 str 타입이 아닙니다: 5
[ValueError] 길이가 1이 아닙니다: 
'ㅠ' -> 모음
'ㅎ' -> 자음

(2) 설명

자음이나 모음이 아닌 문자가 들어온 건 문자열은 맞지만 들어간 '값'이 잘못된 것이기 때문에, 너무 포괄적인 Exception보다는 ValueError 더 적절하다고 판단했습니다. 또한 예외를 처리할 때 부모인 ValueError를 먼저 쓰면 자식인 InvalidJamoError까지 포함되므로, InvalidJamoError를 먼저 처리하도록 순서를 맞췄습니다.